# Sit.3 — Entrenar ConvLSTM 1D + MLP

Predice NO2, SO2, O3 en T+1, T+3, T+7 a partir de secuencias
de 8 embeddings CLIP (generados por sit2_sae_posttrain.ipynb).

Arquitectura: X -> AvgPool(5) -> LSTM(256) -> MLP(128->64) -> y=(3,3)
Loss: MSE enmascarada (ignora NaN en targets)
Dataset: Slucu-0310/geovision-cali-sit3


In [2]:
# @title Dependencias
%pip install -q huggingface_hub numpy pandas tqdm matplotlib
%pip install -q torch lightning

import warnings, os, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
pl.seed_everything(SEED, workers=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


Note: you may need to restart the kernel to use updated packages.


Seed set to 42


Note: you may need to restart the kernel to use updated packages.
Device: cuda


In [3]:
# @title Setup: crear modulos src/ localmente
import sys, os
sys.path.insert(0, ".")

# Crear directorios
os.makedirs("src/modelos", exist_ok=True)
os.makedirs("src/training", exist_ok=True)

# Escribir modelo ConvLSTM1D
with open("src/modelos/convlstm.py", "w") as f:
    f.write("""
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvLSTM1DModel(nn.Module):
    def __init__(self, input_channels=522, seq_len=8, hidden_dim=256,
                 num_layers=2, dropout=0.2, num_horizons=3, num_pollutants=3):
        super().__init__()
        self.input_channels = input_channels
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.lstm = nn.LSTM(input_channels, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
                           bidirectional=False)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_horizons * num_pollutants),
        )
        self._init_weights()

    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if "weight" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
        for module in self.mlp:
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, mode="fan_in", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        N, T, C, H, W = x.shape
        x_r = x.view(N * T, C, H, W)
        x_p = self.avg_pool(x_r).squeeze(-1).squeeze(-1)
        x_p = x_p.view(N, T, C)
        lstm_out, (h_n, _) = self.lstm(x_p)
        h_last = h_n[-1]
        y_flat = self.mlp(h_last)
        return y_flat.view(N, 3, 3)

def masked_mse_loss(y_pred, y_true, weights=None):
    if weights is None:
        weights = torch.tensor([100.0, 10.0, 1.0], device=y_pred.device)
    weights = weights.to(y_pred.device)
    mask = ~torch.isnan(y_true)
    n_valid = mask.sum().float()
    if n_valid < 1.0:
        return torch.tensor(0.0, device=y_pred.device, requires_grad=True)
    loss_per_poll = torch.zeros(3, device=y_pred.device)
    for p in range(3):
        pmask = mask[:, :, p]
        if pmask.sum() > 0:
            loss_per_poll[p] = F.mse_loss(
                y_pred[:, :, p][pmask],
                y_true[:, :, p][pmask],
                reduction="mean",
            )
    weighted_loss = (loss_per_poll * weights).sum() / weights.sum()
    return weighted_loss
""")

# Escribir LightningModule
with open("src/training/lit_convlstm.py", "w") as f:
    f.write("""
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl
from src.modelos.convlstm import ConvLSTM1DModel, masked_mse_loss

class Sit3ConvLSTMDataset(Dataset):
    def __init__(self, x_path, y_path, split_indices=None):
        import numpy as np
        self.X = np.load(x_path, mmap_mode="r")
        self.y = np.load(y_path, mmap_mode="r")
        if split_indices is not None:
            self.X = self.X[split_indices]
            self.y = self.y[split_indices]
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        import numpy as np
        x = torch.from_numpy(self.X[idx].copy()).float()
        y = torch.from_numpy(self.y[idx].copy()).float()
        return {"x": x, "y": y}

import numpy as np

class LitConvLSTM(pl.LightningModule):
    WEIGHTS_DEFAULT = [100.0, 10.0, 1.0]
    def __init__(self, model, lr=1e-3, weight_decay=1e-5, loss_weights=None):
        super().__init__()
        self.save_hyperparameters(ignore=["model"])
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.loss_weights = loss_weights or self.WEIGHTS_DEFAULT
        self._pollutants = ["NO2","SO2","O3"]
        self._horizons = ["T+1","T+3","T+7"]
        self._test_preds = []
        self._test_trues = []
    def forward(self, x):
        return self.model(x)
    def _loss(self, y_pred, y_true):
        w = torch.tensor(self.loss_weights, device=y_pred.device)
        return masked_mse_loss(y_pred, y_true, weights=w)
    def training_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss
    def _log_rmse_per_pollutant(self, y_pred, y_true, prefix):
        for pi, pn in enumerate(self._pollutants):
            vals = y_pred[:, :, pi]
            trues = y_true[:, :, pi]
            m = ~torch.isnan(trues)
            if m.sum() > 0:
                rmse = F.mse_loss(vals[m], trues[m], reduction="mean").sqrt()
                self.log(f"{prefix}/rmse/{pn}", rmse, on_epoch=True)
        for hi, hn in enumerate(self._horizons):
            vals = y_pred[:, hi, :]
            trues = y_true[:, hi, :]
            m = ~torch.isnan(trues)
            if m.sum() > 0:
                rmse = F.mse_loss(vals[m], trues[m], reduction="mean").sqrt()
                self.log(f"{prefix}/rmse/{hn}", rmse, on_epoch=True)
    def validation_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("val/loss", loss, on_epoch=True, prog_bar=True)
        self.log("val/rmse", loss.sqrt(), on_epoch=True, prog_bar=True)
        self._log_rmse_per_pollutant(y_pred, batch["y"], "val")
    def test_step(self, batch, batch_idx):
        y_pred = self.model(batch["x"])
        loss = self._loss(y_pred, batch["y"])
        self.log("test/loss", loss, on_epoch=True)
        self.log("test/rmse", loss.sqrt(), on_epoch=True)
        self._log_rmse_per_pollutant(y_pred, batch["y"], "test")
        self._test_preds.append(y_pred.detach().cpu())
        self._test_trues.append(batch["y"].detach().cpu())
    def on_test_end(self):
        # Acumular todas las predicciones
        all_preds = torch.cat(self._test_preds, dim=0).numpy()
        all_trues = torch.cat(self._test_trues, dim=0).numpy()
        mask = ~np.isnan(all_trues)
        CONV = {"NO2": 5750.0, "SO2": 8008.0, "O3": 6000.0}
        KPIS = {"NO2": 8.0, "SO2": 6.0, "O3": 12.0}
        # Tabla resumen (todo con print, NO self.log)
        print()
        print("=" * 70)
        print("  TABLA RESUMEN — CONVLSTM (SIT. 3)")
        print("=" * 70)
        loss_val = float(masked_mse_loss(torch.from_numpy(all_preds), torch.from_numpy(all_trues)))
        rmse_val = np.sqrt(loss_val)
        print(f"  Loss global (MSE):    {loss_val:.6e}")
        print(f"  RMSE global (mol/m2): {rmse_val:.6f}")
        print()
        print("  Contaminante   RMSE mol/m2      RMSE ug/m3       KPI max  Cumple   R2")
        for pi, pn in enumerate(self._pollutants):
            pmask = mask[:, :, pi]
            y_p = all_preds[:, :, pi][pmask]
            y_t = all_trues[:, :, pi][pmask]
            rmse_p = np.sqrt(np.mean((y_p - y_t) ** 2)) if len(y_p) > 0 else float("nan")
            ug = rmse_p * CONV.get(pn, 1.0)
            kmax = KPIS.get(pn, float("inf"))
            ss_res = np.sum((y_p - y_t) ** 2) if len(y_p) > 1 else 0
            ss_tot = np.sum((y_t - np.mean(y_t)) ** 2) if len(y_t) > 1 else 0
            r2p = 1.0 - ss_res / (ss_tot + 1e-12) if len(y_p) > 1 else float("nan")
            cumple = "SI" if (ug <= kmax) else "NO"
            print(f"  {pn:15s} {rmse_p:.6e}      {ug:8.2f} ug/m3     {kmax:6.1f}     {cumple:8s}  {r2p:.4f}")
        print()
        print("  Horizonte    RMSE mol/m2")
        for hi, hn in enumerate(self._horizons):
            y_ph = all_preds[:, hi, :]
            y_th = all_trues[:, hi, :]
            hmask = ~np.isnan(y_th)
            rmse_h = np.sqrt(np.mean((y_ph[hmask] - y_th[hmask]) ** 2)) if hmask.sum() > 0 else float("nan")
            print(f"  {hn:10s} {rmse_h:.6e}")
        print()
        # R2 global
        if mask.sum() > 1:
            ss_res = np.sum((all_preds[mask] - all_trues[mask]) ** 2)
            ss_tot = np.sum((all_trues[mask] - np.mean(all_trues[mask])) ** 2)
            r2 = 1.0 - ss_res / (ss_tot + 1e-12)
            print(f"  R2 score global: {r2:.4f}")
        else:
            print(f"  R2 score global: NaN")
        print("=" * 70)
        self._test_preds.clear()
        self._test_trues.clear()
    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.model.parameters(),
                                lr=self.lr, weight_decay=self.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=10, min_lr=1e-7)
        return {"optimizer": opt,
                "lr_scheduler": {"scheduler": sched, "monitor": "val/loss"}}
""")

print("Modulos src/ creados localmente")
print("sys.path:", sys.path[:3])


Modulos src/ creados localmente
sys.path: ['.', '/kaggle/working', '/kaggle/lib/kagglegym']


## 1. Descargar dataset de HuggingFace

Dataset: Slucu-0310/geovision-cali-sit3 (X_convlstm.npy + y_convlstm.npy)


In [4]:
# @title Descargar dataset
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except:
        pass

DATA_DIR = Path("/content/dataset_sit3")
if not (DATA_DIR / "X_convlstm.npy").is_file():
    print("Descargando dataset...")
    snapshot_download(
        repo_id="Slucu-0310/geovision-cali-sit3",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
        token=HF_TOKEN,
    )
print("Dataset listo")


Descargando dataset...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Dataset listo


## 2. Cargar datos y crear splits

Se cargan X e y desde .npy y se dividen en 70/15/15
usando los splits del dataset original (heredados de Sit2).


In [5]:
# @title Cargar X, y y crear splits
X = np.load(DATA_DIR / "X_convlstm.npy", mmap_mode="r")
y = np.load(DATA_DIR / "y_convlstm.npy", mmap_mode="r")
print(f"X: {X.shape}  y: {y.shape}")

# Cargar metadatos para splits
import tempfile, os
os.environ["HF_TOKEN"] = HF_TOKEN or ""
import huggingface_hub
meta_path = huggingface_hub.hf_hub_download(
    repo_id="Slucu-0310/geovision-cali-sit2",
    repo_type="dataset",
    filename="metadatos.parquet",
    token=HF_TOKEN,
)
df_meta = pd.read_parquet(meta_path)

# Asignar split a cada secuencia
# Cada secuencia viene de una celda 0.05 con 8 tiles.
# Usamos los splits del primer tile de cada secuencia.
df_meta["celda_lat"] = (df_meta["centroide_lat"] / 0.05).round() * 0.05
df_meta["celda_lon"] = (df_meta["centroide_lon"] / 0.05).round() * 0.05

# Reconstruir asignacion de secuencia a split
# (Esto es aproximado: asumimos que las 1403 secuencias
#  mantienen el mismo orden que en generar_tensor_convlstm.py)
# Para este notebook usamos split aleatorio estratificado.
from sklearn.model_selection import train_test_split

N = len(X)
indices = np.arange(N)
train_idx, tmp_idx = train_test_split(
    indices, test_size=0.3, random_state=SEED
)
val_idx, test_idx = train_test_split(
    tmp_idx, test_size=0.5, random_state=SEED
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")


X: (1403, 8, 522, 5, 5)  y: (1403, 3, 3)


metadatos.parquet:   0%|          | 0.00/335k [00:00<?, ?B/s]

Train: 982, Val: 210, Test: 211


In [6]:
# @title Crear datasets y dataloaders
from src.training.lit_convlstm import Sit3ConvLSTMDataset, LitConvLSTM
from src.modelos.convlstm import ConvLSTM1DModel
from lightning.pytorch.loggers import CSVLogger

BATCH_SIZE = 32

ds_train = Sit3ConvLSTMDataset(
    str(DATA_DIR / "X_convlstm.npy"),
    str(DATA_DIR / "y_convlstm.npy"),
    split_indices=train_idx.tolist(),
)
ds_val = Sit3ConvLSTMDataset(
    str(DATA_DIR / "X_convlstm.npy"),
    str(DATA_DIR / "y_convlstm.npy"),
    split_indices=val_idx.tolist(),
)
ds_test = Sit3ConvLSTMDataset(
    str(DATA_DIR / "X_convlstm.npy"),
    str(DATA_DIR / "y_convlstm.npy"),
    split_indices=test_idx.tolist(),
)

dl_train = DataLoader(ds_train, BATCH_SIZE, shuffle=True, num_workers=0)
dl_val = DataLoader(ds_val, BATCH_SIZE, shuffle=False, num_workers=0)
dl_test = DataLoader(ds_test, BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches: train={len(dl_train)}, val={len(dl_val)}, test={len(dl_test)}")


Batches: train=31, val=7, test=7


## 3. Crear modelo y entrenar

ConvLSTM 1D: AvgPool -> LSTM(256, 2 capas) -> MLP(128->64) -> 9 salidas


In [7]:
# @title Crear modelo
model = ConvLSTM1DModel(
    input_channels=522,
    seq_len=8,
    hidden_dim=256,
    num_layers=2,
    dropout=0.2,
)
total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo creado. Parametros: {total_params:,}")


Modelo creado. Parametros: 1,366,793


In [8]:
# @title Entrenar
RUN_DIR = Path("/content/runs/sit3_convlstm")
RUN_DIR.mkdir(parents=True, exist_ok=True)

lit_model = LitConvLSTM(model, lr=1e-3, weight_decay=1e-5)

callbacks = [
    pl.callbacks.EarlyStopping(
        monitor="val/loss",
        patience=40,
        mode="min",
    ),
    pl.callbacks.ModelCheckpoint(
        dirpath=str(RUN_DIR),
        filename="best",
        monitor="val/loss",
        mode="min",
        save_top_k=1,
    ),
]

trainer = pl.Trainer(
    max_epochs=400,
    accelerator="auto",
    devices=1,
    callbacks=callbacks,
    logger=CSVLogger(save_dir=str(RUN_DIR / "lightning_logs")),
    log_every_n_steps=5,
    gradient_clip_val=1.0,
    default_root_dir=str(RUN_DIR),
)

trainer.fit(lit_model, dl_train, dl_val)

print(f"Entrenamiento completo. Mejor checkpoint en {RUN_DIR}")


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Norm std (por contaminante): tensor([1.7249e-05, 2.8755e-04, 5.4255e-03])


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ConvLSTM1DModel │  1.4 M │ train │     0 │
└───┴───────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5.467                                                                      
Modules in train mode: 10                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Entrenamiento completo. Mejor checkpoint en /content/runs/sit3_convlstm


## 4. Evaluacion en test


In [9]:
# @title Evaluar en test
ckpt_path = str(RUN_DIR / "best.pt") if (RUN_DIR / "best.pt").is_file() else None
# Buscar el mejor checkpoint
ckpts = sorted(RUN_DIR.glob("best*"), key=os.path.getmtime, reverse=True)
if ckpts:
    ckpt_path = str(ckpts[0])
    print(f"Usando checkpoint: {ckpt_path}")
    trainer.test(lit_model, dl_test, ckpt_path=ckpt_path)
else:
    print("Checkpoint no encontrado, evaluando con ultimo modelo")
    trainer.test(lit_model, dl_test)


Restoring states from the checkpoint path at /content/runs/sit3_convlstm/best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /content/runs/sit3_convlstm/best.ckpt


Output()

Usando checkpoint: /content/runs/sit3_convlstm/best.ckpt


======================================================================

TABLA RESUMEN — CONVLSTM (SIT. 3)

...

T+3        3.326043e-03

T+7        3.425305e-03

R2 score global: 0.9964

======================================================================

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/loss         │    0.9376901388168335     │
│         test/rmse         │     0.967822253704071     │
│       test/rmse/NO2       │   1.765958768373821e-05   │
│       test/rmse/O3        │    0.00557215791195631    │
│       test/rmse/SO2       │  0.00023315168800763786   │
│       test/rmse/T+1       │   0.003188218455761671    │
│       test/rmse/T+3       │   0.003311537904664874    │
│       test/rmse/T+7       │   0.0034100371412932873   │
└───────────────────────────┴───────────────────────────┘

## 5. Curvas de entrenamiento


In [10]:
# @title Graficar loss y RMSE
logs = trainer.logged_metrics if hasattr(trainer, "logged_metrics") else {}
if logs:
    print("Metricas finales:")
    for k, v in sorted(logs.items()):
        if isinstance(v, torch.Tensor):
            print(f"  {k}: {v.item():.6f}")
        else:
            print(f"  {k}: {v:.6f}")

# Cargar historial desde lightning_logs
import glob as gl
csv_files = gl.glob(str(RUN_DIR / "lightning_logs" / "version_*" / "metrics.csv"))
if csv_files:
    df_log = pd.read_csv(csv_files[0])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    if "train/loss_epoch" in df_log.columns:
        axes[0].plot(df_log["train/loss_epoch"], label="train", alpha=0.7)
    if "val/loss" in df_log.columns:
        axes[0].plot(df_log["val/loss"], label="val", alpha=0.7)
    axes[0].set_xlabel("Epoca")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("Loss de entrenamiento")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # RMSE
    for p in ["NO2", "SO2", "O3"]:
        col = "val/rmse/" + p
        if col in df_log.columns:
            axes[1].plot(df_log[col], label=p, alpha=0.7)
    axes[1].set_xlabel("Epoca")
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("RMSE por contaminante (val)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(str(RUN_DIR / "training_curves.png"), dpi=150)
    plt.show()
    print(f"Curvas guardadas en {RUN_DIR / 'training_curves.png'}")
else:
    print("No se encontraron logs de entrenamiento.")


Metricas finales:
  test/loss: 0.937690
  test/rmse: 0.967822
  test/rmse/NO2: 0.000018
  test/rmse/O3: 0.005572
  test/rmse/SO2: 0.000233
  test/rmse/T+1: 0.003188
  test/rmse/T+3: 0.003312
  test/rmse/T+7: 0.003410
No se encontraron logs de entrenamiento.


## 6. Resumen de metricas

| KPI | Minimo | Medicion |
|-----|--------|----------|
| RMSE LOO-CV NO2 (T+1) | <= 8 ug/m3 | (test) |
| RMSE LOO-CV SO2 (T+1) | <= 6 ug/m3 | (test) |
| RMSE LOO-CV O3 (T+1) | <= 12 ug/m3 | (test) |
| R2 LOO-CV (promedio) | >= 0.55 | (test) |
